https://drive.google.com/drive/folders/1zWF80Yo2-pDa0TjuH4STXf8P5t40N8Ak

In [ ]:
!pip -q install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()

In [ ]:
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+



In [ ]:
sales_base_df = (
spark.range(1, 100001)
.withColumnRenamed("id", "order_id")
.withColumn(
"order_date",
F.date_add(
F.to_date(F.lit("2026-01-01")),
(F.col("order_id") % 180).cast("int")
)
)
.withColumn(
"product_id",
F.concat(
F.lit("P"),
F.lpad(
((F.col("order_id") % 6) + 1).cast("string"),
2, "0"
)
)
)
.withColumn(
"store_id",
F.concat(
F.lit("S"),
F.lpad(
((F.floor(F.col("order_id") / 3) % 4) + 1)
.cast("string"),
2, "0"
)
)
)
.withColumn(
"quantity",
((F.col("order_id") % 8) + 1).cast("int")
)
.withColumn(
"discount_pct",
F.when(
F.col("order_id") % 19 == 0,
F.lit(None).cast("int")
).otherwise(
((F.floor(F.col("order_id") / 5) % 4) * 5)
.cast("int")
)
)
)

In [ ]:
sales_df = (
sales_base_df
.withColumn(
"order_status",
F.when(F.col("order_id") % 11 == 0, "Cancelled")
.when(F.col("order_id") % 7 == 0, "Returned")
.otherwise("Completed")
)
.withColumn(
"payment_method",
F.when(F.col("order_id") % 3 == 0, "UPI")
.when(F.col("order_id") % 3 == 1, "Card")
.otherwise("Cash")
)
.withColumn(
"salesperson_id",
F.when(
F.col("order_id") % 17 == 0,
F.lit(None).cast("string")
).otherwise(
F.concat(
F.lit("EMP"),
F.lpad(
((F.col("order_id") % 12) + 1).cast("string"),
2, "0"
)
)
)
)
.join(
products_df.select(
"product_id",
F.col("list_price").alias("unit_price")
),
"product_id", "left"
)
.select(
"order_id", "order_date", "store_id",
"product_id", "quantity", "unit_price",
"discount_pct", "order_status",
"payment_method", "salesperson_id"
)
)
print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |C

**Practice Exercises**

Part A | DataFrame transformations

In [ ]:
sales_df.select("order_id","product_id","quantity","unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [ ]:
sales_df.filter((F.col("order_status")=="Completed")&(F.col("quantity")>5)).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [ ]:
sales_df.withColumn("gross_revenue",F.col("quantity")*F.col("unit_price")).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|       136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|       102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|  

In [ ]:
sales_df.withColumnRenamed("order_status","transaction_status")\
  .drop("payment_method").show()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|
|      14|2026-01-1

Part B | Aggregations and joins

In [ ]:
sales_df.groupBy("product_id").agg(F.sum(F.col("quantity")).alias("total_quantity")).show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P04|         83334|
|       P03|         66667|
|       P05|         66665|
|       P06|         83330|
|       P02|         83336|
|       P01|         66668|
+----------+--------------+



In [ ]:
sales_df.join(products_df,"product_id","inner")\
  .groupBy("category").agg(F.sum(F.col("quantity")*F.col("unit_price")).alias("total_gross_revenue")).show()


+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|     Office|         1416678000|
| Networking|          299992500|
|Electronics|         7166704000|
|Accessories|          233334500|
+-----------+-------------------+



In [ ]:
sales_df.join(stores_df,"store_id","inner")\
  .groupBy("city").agg(
      F.count("*").alias("total_transactions")
  ).show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|   Mumbai|             24999|
|    Delhi|             24999|
|Bengaluru|             25001|
|Hyderabad|             25001|
+---------+------------------+



In [ ]:
sales_df.join(products_df,"product_id","inner")\
  .join(stores_df,"store_id","inner")\
  .filter(F.col("order_status")=="Completed")\
  .withColumn("gross_revenue",F.col("quantity")*F.col("unit_price"))\
  .groupBy("product_id","product_name")\
  .agg(F.sum("gross_revenue").alias("total_gross_revenue"))\
  .sort(F.col("total_gross_revenue").desc()).show(5)

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+
only showing top 5 rows


Part C | Null handling and conditions

In [ ]:
sales_df.select(
    F.sum(F.when(F.col("discount_pct").isNull(),1).otherwise(0)).alias("null_discount"),
    F.sum(F.when(F.col("salesperson_id").isNull(),1).otherwise(0)).alias("null_salesperson")
).show()

+-------------+----------------+
|null_discount|null_salesperson|
+-------------+----------------+
|         5263|            5882|
+-------------+----------------+



In [ ]:
sales_df.na.fill({"discount_pct":0}).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [ ]:
sales_df.na.drop(subset=["salesperson_id"]).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [ ]:
sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity")<=3,"Small")
    .when(F.col("quantity")<=6,"Medium")
    .otherwise("Large")
).select("order_id","quantity","order_size").show()

+--------+--------+----------+
|order_id|quantity|order_size|
+--------+--------+----------+
|       3|       4|    Medium|
|       9|       2|     Small|
|      15|       8|     Large|
|      21|       6|    Medium|
|       2|       3|     Small|
|       8|       1|     Small|
|      14|       7|     Large|
|      20|       5|    Medium|
|       4|       5|    Medium|
|      10|       3|     Small|
|      16|       1|     Small|
|       5|       6|    Medium|
|      11|       4|    Medium|
|      17|       2|     Small|
|       1|       2|     Small|
|       7|       8|     Large|
|      13|       6|    Medium|
|      19|       4|    Medium|
|       6|       7|     Large|
|      12|       5|    Medium|
+--------+--------+----------+
only showing top 20 rows


Part D | Window and date functions

In [ ]:
sales_df.join(products_df,"product_id","inner")\
  .withColumn("gross_revenue",F.col("quantity")*F.col("unit_price"))\
  .groupBy("category", "product_id", "product_name")\
  .agg(F.sum("gross_revenue").alias("total_gross_revenue"))\
  .withColumn("rank",F.dense_rank().over(Window.partitionBy("category").orderBy(F.col("total_gross_revenue").desc()))).show()

+-----------+----------+------------+-------------------+----+
|   category|product_id|product_name|total_gross_revenue|rank|
+-----------+----------+------------+-------------------+----+
|Accessories|       P03|  Headphones|          233334500|   1|
|Electronics|       P01|      Laptop|         4000080000|   1|
|Electronics|       P06|      Tablet|         1999920000|   2|
|Electronics|       P02|     Monitor|         1166704000|   3|
| Networking|       P05|      Router|          299992500|   1|
|     Office|       P04|     Printer|         1416678000|   1|
+-----------+----------+------------+-------------------+----+



In [ ]:

df14 = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)

window14 = (
    Window.partitionBy("store_id")
    .orderBy("order_date", "order_id")
)

df14.withColumn(
    "running_revenue",
    F.sum("gross_revenue").over(window14)
).select(
    "store_id",
    "order_id",
    "order_date",
    "gross_revenue",
    "running_revenue"
).show()


+--------+--------+----------+-------------+---------------+
|store_id|order_id|order_date|gross_revenue|running_revenue|
+--------+--------+----------+-------------+---------------+
|     S01|     180|2026-01-01|       300000|         300000|
|     S01|     360|2026-01-01|        60000|         360000|
|     S01|     540|2026-01-01|       300000|         660000|
|     S01|     720|2026-01-01|        60000|         720000|
|     S01|     900|2026-01-01|       300000|        1020000|
|     S01|    1080|2026-01-01|        60000|        1080000|
|     S01|    1260|2026-01-01|       300000|        1380000|
|     S01|    1440|2026-01-01|        60000|        1440000|
|     S01|    1620|2026-01-01|       300000|        1740000|
|     S01|    1800|2026-01-01|        60000|        1800000|
|     S01|    1980|2026-01-01|       300000|        2100000|
|     S01|    2160|2026-01-01|        60000|        2160000|
|     S01|    2340|2026-01-01|       300000|        2460000|
|     S01|    2520|2026-

In [ ]:
sales_df.filter(F.col("order_status")=="Completed")\
  .withColumn("month",F.month(F.col("order_date")))\
  .withColumn("gross_revenue",F.col("quantity")*F.col("unit_price"))\
  .groupBy("month")\
  .agg(F.sum("gross_revenue").alias("total_gross_revenue")).orderBy("month").show()

+-----+-------------------+
|month|total_gross_revenue|
+-----+-------------------+
|    1|         1278728500|
|    2|         1027585500|
|    3|         1212087500|
|    4|         1218921000|
|    5|         1277560500|
|    6|         1089027500|
+-----+-------------------+



Part E | Lazy execution

In [ ]:
df16 = sales_df.filter(
    F.col("quantity") > 5
)
print("Filter transformation created")

df16 = df16.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)
print("withColumn transformation created")

df16 = df16.select(
    "order_id", "product_id", "gross_revenue"
)
print("Select transformation created")

df16.show()


Filter transformation created
withColumn transformation created
Select transformation created
+--------+----------+-------------+
|order_id|product_id|gross_revenue|
+--------+----------+-------------+
|      15|       P04|       136000|
|      21|       P04|       102000|
|      39|       P04|       136000|
|      45|       P04|       102000|
|      14|       P03|        24500|
|      38|       P03|        24500|
|      22|       P05|        31500|
|      46|       P05|        31500|
|       5|       P06|       144000|
|      23|       P06|       192000|
|      29|       P06|       144000|
|      47|       P06|       192000|
|      53|       P06|       144000|
|       7|       P02|       112000|
|      13|       P02|        84000|
|      31|       P02|       112000|
|      37|       P02|        84000|
|      55|       P02|       112000|
|       6|       P01|       420000|
|      30|       P01|       420000|
+--------+----------+-------------+
only showing top 20 rows


In [ ]:
df17 = sales_df.groupBy("product_id").agg(
    F.sum("quantity").alias("total_quantity")
)

df17.explain()

result = df17.collect()

print(result)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(quantity#35)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(quantity#35)])
      +- Project [product_id#33, quantity#35]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5172]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=5173]
                  +- Project [product_id#0]
                     +- Filter isnotnull(product_id#0)
                        +- Scan ExistingRDD

Part F | Caching and persistence

In [ ]:
df18=sales_df.filter(F.col("order_status")=="Completed").cache()
start=time.time()
print("Count 1:",df18.count())
print("Time:",time.time()-start)
start=time.time()
print("Count 2:",df18.count())
print("Time:",time.time()-start)
df18.unpersist()

Count 1: 77923
Time: 2.6156423091888428
Count 2: 77923
Time: 3.8164923191070557


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [ ]:
df19=sales_df.filter(F.col("quantity")>5).persist(StorageLevel.MEMORY_AND_DISK)
print("Count:",df19.count())
df19.groupBy("product_id").count().show()
df19.unpersist()

Count: 37500
+----------+-----+
|product_id|count|
+----------+-----+
|       P04| 8333|
|       P03| 4167|
|       P05| 4166|
|       P06| 8333|
|       P02| 8334|
|       P01| 4167|
+----------+-----+



DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [ ]:
df20=sales_df.groupBy("product_id").agg(
    F.sum("quantity").alias("total_quantity")
).cache()
df20.explain()
df20.show()
df20.unpersist()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [product_id#33, total_quantity#2343L]
      +- InMemoryRelation [product_id#33, total_quantity#2343L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- HashAggregate(keys=[product_id#33], functions=[sum(quantity#35)])
                  +- HashAggregate(keys=[product_id#33], functions=[partial_sum(quantity#35)])
                     +- Project [product_id#33, quantity#35]
                        +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
                           :- Sort [product_id#33 ASC NULLS FIRST], false, 0
                           :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5820]
                           :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
                           :  

DataFrame[product_id: string, total_quantity: bigint]

Part G | Repartition and coalesce

In [ ]:
print(sales_df.rdd.getNumPartitions())
df21=sales_df.repartition(8)
print(df21.rdd.getNumPartitions())

2
8


In [ ]:
df22=sales_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
df22.groupBy("partition_id").count().orderBy("partition_id").show()

+------------+------+
|partition_id| count|
+------------+------+
|           0|100000|
+------------+------+



In [ ]:
df23 = sales_df.repartition(
    6, "store_id"
)

print("Partitions:", df23.rdd.getNumPartitions())

df23.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id"
).agg(
    F.collect_set("store_id").alias("stores"),
    F.count("*").alias("records")
).orderBy("partition_id").show(truncate=False)


Partitions: 6
+------------+------+-------+
|partition_id|stores|records|
+------------+------+-------+
|1           |[S03] |24999  |
|3           |[S02] |25001  |
|4           |[S01] |25001  |
|5           |[S04] |24999  |
+------------+------+-------+



In [ ]:
df24=sales_df.repartition(12)
print("Start:",df24.rdd.getNumPartitions())
coalesce_df=df24.coalesce(4)
print("Coalesce df:",coalesce_df.rdd.getNumPartitions())
print("Execution plan:",coalesce_df.explain())
repartition_df=df24.repartition(4)
print("Repartition df:",repartition_df.rdd.getNumPartitions())
print("Execution plan:",repartition_df.explain())

Start: 12
Coalesce df: 4
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- Coalesce 4
      +- ShuffleQueryStage 3
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=7558]
            +- *(3) Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
               +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                  :- AQEShuffleRead local
                  :  +- ShuffleQueryStage 0
                  :     +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=7442]
                  :        +- *(1) Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, concat(S, lpad(cast(((FLOOR((cast(id#30L as double

In [ ]:
sales_df.repartition(4).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/sales_output")


**Final Challenge**

In [ ]:
altered_df=sales_df.filter(F.col("order_status")=="Completed")

In [ ]:
altered_df=altered_df.na.fill({"discount_pct":0})

In [ ]:
altered_df=altered_df.join(products_df,"product_id","inner").join(stores_df,"store_id","inner")


In [ ]:
altered_df=altered_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "discounted_revenue",
    F.col("gross_revenue") * (1 - F.col("discount_pct") / 100)
)

In [ ]:
altered_df = altered_df.groupBy(
    "region", "category"
).agg(
    F.sum("gross_revenue").alias("total_gross_revenue"),
    F.sum("discounted_revenue").alias("total_revenue"),
    F.count("*").alias("transaction_count")
)


In [ ]:
ranking_window = Window.partitionBy(
    "region"
).orderBy(
    F.desc("total_revenue")
)

altered_df = altered_df.withColumn(
    "category_rank",
    F.dense_rank().over(ranking_window))

In [ ]:
altered_df.cache()

DataFrame[region: string, category: string, total_gross_revenue: bigint, total_revenue: double, transaction_count: bigint, category_rank: int]

In [ ]:
altered_df.show()

+------+-----------+-------------------+-------------+-----------------+-------------+
|region|   category|total_gross_revenue|total_revenue|transaction_count|category_rank|
+------+-----------+-------------------+-------------+-----------------+-------------+
| South|Electronics|         2156236000|  2.0242794E9|            19484|            1|
| South|     Office|          662524000|   6.060058E8|             6495|            2|
| South|Accessories|          113631000| 1.05018025E8|             6494|            3|
| South| Networking|           87669000|  8.2684125E7|             6494|            4|
|  West|Electronics|         2493348000|  2.2992404E9|            12985|            1|
|  West|Accessories|           68169500|    6.42985E7|             6493|            2|
| North|Electronics|          934848000|   8.551824E8|             6492|            1|
| North|     Office|          441388000|    4.12114E8|             6492|            2|
| North| Networking|          146097000| 1.

In [ ]:
altered_df.repartition(2).write \
    .mode("overwrite") \
    .parquet("/content/final_sales_output")

altered_df.unpersist()


DataFrame[region: string, category: string, total_gross_revenue: bigint, total_revenue: double, transaction_count: bigint, category_rank: int]